# SmishGuard — DistilBERT Fine-Tuning for SMS Fraud Detection

**Capstone Project · Ashesi University**

This notebook fine-tunes a **DistilBERT** model for binary SMS classification
(Safe vs Threat) using Ghanaian telecom SMS messages collected via survey.

### Pipeline
1. Data Loading & Parsing
2. Data Cleaning & PII check
3. Data Augmentation (class balancing)
4. Tokenization & Dataset Splitting
5. Fine-Tuning (binary: Safe vs Threat)
6. Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)
7. Export to TFLite for Android deployment

> **Note:** The model performs binary classification (Safe=0, Threat=1).
> On-device, a rule-based layer then distinguishes **Spam** vs **Fraud**
> among Threat predictions, alongside sender-whitelist and contact-matching.

In [ ]:
# Fix for Python 3.12 + TF 2.16+ on Google Colab
# Must set BEFORE any TF import — forces TF to use Keras 2 API
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

# transformers 4.37.2 = last version with TF model classes (removed in 4.38+)
# tf-keras = Keras 2 compatibility layer for TF 2.16+
!pip install -q 'transformers==4.37.2' tf-keras datasets accelerate scikit-learn matplotlib seaborn nlpaug

---
## Phase 1 — Load & Parse the Ghana SMS Dataset

The CSV has two columns:
- `Label`: One of `Safe`, `Spam`, or `Fraud`
- `Text`: Format is `SenderName: "actual message text"`

The sender name stays in the model input (per project design) so DistilBERT
can learn associations between known senders and safety.

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

# ── Path to your CSV in Google Drive ──
# Adjust the path if your file is in a different folder
csv_filename = '/content/drive/MyDrive/Ghana_sms_text.csv'

# ── Parse manually because of nested quoting ──
rows = []
with open(csv_filename, 'r', encoding='utf-8') as f:
    header = f.readline().strip()  # skip header
    for line in f:
        line = line.strip()
        if not line:
            continue
        # Split on first comma only — Label never contains commas
        idx = line.index(',')
        label = line[:idx].strip()
        text = line[idx+1:].strip()
        # Remove outer CSV quotes if present
        if text.startswith('"') and text.endswith('"'):
            text = text[1:-1]
            text = text.replace('""', '"')  # un-escape doubled quotes
        rows.append({'label': label, 'text': text})

df = pd.DataFrame(rows)
print(f"\nLoaded {len(df)} messages")
print(f"\nLabel distribution:")
print(df['label'].value_counts())

In [ ]:
# Preview messages by label
for label in ['Safe', 'Spam', 'Fraud']:
    subset = df[df['label'] == label]
    print(f"\n{'='*60}")
    print(f"  {label.upper()} — {len(subset)} messages")
    print(f"{'='*60}")
    for i, row in subset.head(3).iterrows():
        print(f"  [{i}] {row['text'][:120]}{'...' if len(row['text']) > 120 else ''}")

---
## Phase 2 — Data Cleaning

- Normalize whitespace
- Verify PII redaction ([NAME], [AMOUNT], [NUMBER], [REFERENCE] placeholders)
- Strip extraneous characters
- Remove exact duplicates

In [ ]:
def clean_text(text):
    # Normalize SMS text for model training.
    text = str(text).strip()
    # Collapse multiple whitespace / newlines into single space
    text = re.sub(r'\s+', ' ', text)
    return text

df['text'] = df['text'].apply(clean_text)

# Check PII status — look for any un-redacted Ghanaian phone numbers
pii_pattern = re.compile(r'\b0[2345]\d[\s-]?\d{3}[\s-]?\d{4}\b')
pii_hits = df[df['text'].apply(lambda t: bool(pii_pattern.search(t)))]
if len(pii_hits) > 0:
    print(f"⚠ Found {len(pii_hits)} messages with possible un-redacted phone numbers:")
    for i, row in pii_hits.iterrows():
        match = pii_pattern.search(row['text'])
        print(f"  [{i}] ...{row['text'][max(0,match.start()-20):match.end()+20]}...")
    print("\nThese should be reviewed before publication, but we proceed with training.")
else:
    print("✓ No un-redacted Ghanaian phone numbers detected.")

# Remove exact duplicates
before = len(df)
df = df.drop_duplicates(subset='text').reset_index(drop=True)
print(f"\nDuplicates removed: {before - len(df)}")
print(f"Final dataset: {len(df)} messages")
print(df['label'].value_counts())

---
## Phase 3 — Binary Labels & Augmentation

**Classification strategy (hybrid):**
- The model performs **binary classification**: Safe (0) vs Threat (1)
- Spam and Fraud are both mapped to Threat (1) for training
- On the Android device, a rule-based layer then distinguishes Spam vs Fraud

**Augmentation:** Since Threat class (~18 msgs) is underrepresented vs Safe (~42 msgs),
we apply synonym-replacement augmentation to balance the classes.

In [ ]:
# Map to binary: Safe=0, Threat=1 (Spam + Fraud merged)
label_map = {'Safe': 0, 'Spam': 1, 'Fraud': 1}
df['binary_label'] = df['label'].map(label_map)

print("Binary label distribution:")
print(df['binary_label'].value_counts().rename({0: 'Safe (0)', 1: 'Threat (1)'}))

safe_count = (df['binary_label'] == 0).sum()
threat_count = (df['binary_label'] == 1).sum()
print(f"\nImbalance ratio: {safe_count/threat_count:.1f}:1 (Safe:Threat)")
print(f"Target: augment Threat class from {threat_count} → ~{safe_count} messages")

In [ ]:
import nlpaug.augmenter.word as naw

# Synonym replacement augmenter
# This replaces words with synonyms to create natural variations
aug = naw.SynonymAug(aug_src='wordnet', aug_max=3)

def augment_texts(texts, target_count, seed=42):
    # Augment a list of texts to reach target_count via synonym replacement.
    np.random.seed(seed)
    augmented = list(texts)  # start with originals
    source_texts = list(texts)

    while len(augmented) < target_count:
        # Pick a random original text to augment
        original = source_texts[np.random.randint(0, len(source_texts))]
        try:
            new_text = aug.augment(original)
            if isinstance(new_text, list):
                new_text = new_text[0]
            augmented.append(new_text)
        except Exception:
            # If augmentation fails, add the original with minor variation
            augmented.append(original)
    return augmented

# Separate by class
safe_df = df[df['binary_label'] == 0].copy()
threat_df = df[df['binary_label'] == 1].copy()

# Augment threat class to match safe class size
target = len(safe_df)
print(f"Augmenting Threat class: {len(threat_df)} → {target} messages...")

augmented_threat_texts = augment_texts(
    threat_df['text'].tolist(),
    target_count=target
)

# Build augmented threat dataframe
aug_threat_df = pd.DataFrame({
    'text': augmented_threat_texts,
    'label': ['Threat'] * len(augmented_threat_texts),
    'binary_label': [1] * len(augmented_threat_texts)
})

# Combine
safe_df['label'] = 'Safe'
balanced_df = pd.concat([safe_df[['text', 'label', 'binary_label']],
                          aug_threat_df], ignore_index=True)
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced dataset: {len(balanced_df)} messages")
print(balanced_df['binary_label'].value_counts().rename({0: 'Safe (0)', 1: 'Threat (1)'}))

In [ ]:
# Preview some augmented threat messages vs originals
print("=== Original Threat Messages (sample) ===")
for i, row in threat_df.head(3).iterrows():
    print(f"  {row['text'][:100]}...")

print(f"\n=== Augmented Threat Messages (sample) ===")
# Show only the generated ones (after the originals)
for text in augmented_threat_texts[len(threat_df):len(threat_df)+3]:
    print(f"  {text[:100]}...")

---
## Phase 4 — Train/Validation/Test Split & Tokenization

- **70% train** / **15% validation** / **15% test** (stratified)
- Tokenization with `DistilBertTokenizerFast`, max length 128 tokens
  (SMS messages are short — 128 is more than enough)

In [ ]:
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast
from datasets import Dataset

# Stratified split
train_df, temp_df = train_test_split(
    balanced_df, test_size=0.30, stratify=balanced_df['binary_label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['binary_label'], random_state=42
)

print(f"Splits:")
print(f"  Train:      {len(train_df)} ({len(train_df)/len(balanced_df):.0%})")
print(f"  Validation: {len(val_df)} ({len(val_df)/len(balanced_df):.0%})")
print(f"  Test:       {len(test_df)} ({len(test_df)/len(balanced_df):.0%})")

# Load tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# Convert to HuggingFace Datasets
def make_dataset(dataframe):
    ds = Dataset.from_pandas(dataframe[['text', 'binary_label']].rename(
        columns={'binary_label': 'label'}).reset_index(drop=True))
    return ds

train_dataset = make_dataset(train_df)
val_dataset = make_dataset(val_df)
test_dataset = make_dataset(test_df)

# Tokenize
def tokenize_fn(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

print("\nTokenizing...")
train_dataset = train_dataset.map(tokenize_fn, batched=True, batch_size=64)
val_dataset = val_dataset.map(tokenize_fn, batched=True, batch_size=64)
test_dataset = test_dataset.map(tokenize_fn, batched=True, batch_size=64)

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("✓ Tokenization complete")
print(f"  Sample shape: {train_dataset[0]['input_ids'].shape}")

---
## Phase 5 — Fine-Tune DistilBERT

- **Model**: `distilbert-base-uncased` + binary classification head
- **Class weights**: Computed from training set to handle any residual imbalance
- **Epochs**: 4 (small dataset needs more passes but risk of overfitting)
- **Learning rate**: 2e-5 (standard for BERT fine-tuning)
- **Warmup**: 10% of steps (gradually increase LR to prevent early instability)

In [ ]:
import torch
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

# Load model with 2-class head
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2
)

print(f"Parameters: {model.num_parameters():,}")

# Class weights
train_labels = train_df['binary_label'].values
class_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Class weights: Safe={class_weights[0]:.3f}, Threat={class_weights[1]:.3f}")

In [ ]:
# Custom trainer with class-weighted loss
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

training_args = TrainingArguments(
    output_dir='./smishguard_training',
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to='none',
    fp16=torch.cuda.is_available(),
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print(f"Training config:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  LR: {training_args.learning_rate}")
print(f"  Train samples: {len(train_dataset)}")
print(f"  Val samples: {len(val_dataset)}")
print()

train_result = trainer.train()
print(f"\n✓ Training complete — Loss: {train_result.training_loss:.4f}")

---
## Phase 6 — Evaluation

Test set metrics. For a fraud detection app:
- **Recall** is critical — missing a real threat is worse than a false alarm
- **Precision** matters for user trust — too many false alarms and users disable the app

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
labels = test_dataset['label'].numpy()

print("=" * 55)
print("  CLASSIFICATION REPORT — Test Set")
print("=" * 55)
print(classification_report(labels, preds, target_names=['Safe', 'Threat'], digits=4))

m = predictions.metrics
print(f"Summary: Acc={m['test_accuracy']:.3f}  P={m['test_precision']:.3f}  "
      f"R={m['test_recall']:.3f}  F1={m['test_f1']:.3f}")

In [ ]:
# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(labels, preds)
ConfusionMatrixDisplay(cm, display_labels=['Safe', 'Threat']).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Confusion Matrix (Counts)')

cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
ConfusionMatrixDisplay(cm_pct, display_labels=['Safe', 'Threat']).plot(ax=axes[1], cmap='Blues', values_format='.1%')
axes[1].set_title('Confusion Matrix (%)')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Neg (safe→safe): {tn}   False Pos (safe→threat): {fp}")
print(f"False Neg (threat→safe): {fn}   True Pos (threat→threat): {tp}")
print(f"False alarm rate: {fp/(fp+tn):.1%}   Miss rate: {fn/(fn+tp):.1%}")

In [ ]:
# Training history
history = trainer.state.log_history
train_losses = [(h['step'], h['loss']) for h in history if 'loss' in h and 'eval_loss' not in h]
eval_entries = [h for h in history if 'eval_loss' in h]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if train_losses:
    steps, losses = zip(*train_losses)
    axes[0].plot(steps, losses, 'b-', alpha=0.7)
    axes[0].set_title('Training Loss'); axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)

if eval_entries:
    epochs = range(1, len(eval_entries) + 1)
    axes[1].plot(epochs, [e['eval_f1'] for e in eval_entries], 'g-o', label='F1')
    axes[1].plot(epochs, [e['eval_precision'] for e in eval_entries], 'b-s', label='Precision')
    axes[1].plot(epochs, [e['eval_recall'] for e in eval_entries], 'r-^', label='Recall')
    axes[1].set_title('Validation Metrics per Epoch'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(True, alpha=0.3); axes[1].set_ylim(0.5, 1.05)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Phase 7 — Export for Android (TFLite)

1. Save PyTorch model
2. Convert to TensorFlow
3. Convert to TFLite with **float16 quantization** (~50% size reduction)
4. Export vocabulary file for on-device tokenization
5. Export `safe_senders.txt` list extracted from training data

In [ ]:
# Save fine-tuned model
MODEL_DIR = './smishguard_model'
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

import os
total = sum(os.path.getsize(os.path.join(MODEL_DIR, f))
            for f in os.listdir(MODEL_DIR) if os.path.isfile(os.path.join(MODEL_DIR, f)))
print(f"Model saved to {MODEL_DIR}/ — {total/1024/1024:.1f} MB")

In [ ]:
# Convert to TFLite
import tensorflow as tf
from transformers import TFDistilBertForSequenceClassification

print("Loading as TensorFlow model...")
tf_model = TFDistilBertForSequenceClassification.from_pretrained(MODEL_DIR, from_pt=True)
print("✓ TF model loaded")

MAX_LEN = 128

@tf.function(input_signature=[
    tf.TensorSpec([1, MAX_LEN], tf.int32, name='input_ids'),
    tf.TensorSpec([1, MAX_LEN], tf.int32, name='attention_mask'),
])
def serving_fn(input_ids, attention_mask):
    output = tf_model(input_ids=input_ids, attention_mask=attention_mask)
    return {'probabilities': tf.nn.softmax(output.logits, axis=-1)}

print("Converting to TFLite (float16 quantization)...")
converter = tf.lite.TFLiteConverter.from_concrete_functions(
    [serving_fn.get_concrete_function()])
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

TFLITE_PATH = './smishguard_model.tflite'
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize(TFLITE_PATH) / 1024 / 1024
print(f"\n✓ TFLite model: {TFLITE_PATH} ({size_mb:.1f} MB)")

In [ ]:
# Extract safe sender names from training data for the Android whitelist
import json

# Read original (non-augmented) data to get sender names
safe_senders = set()
original_safe = df[df['label'] == 'Safe']['text']
for text in original_safe:
    # Format: "SenderName: ..."
    match = re.match(r'^([^:]+):', text)
    if match:
        sender = match.group(1).strip()
        safe_senders.add(sender)

# Write safe_senders.txt
with open('./safe_senders.txt', 'w') as f:
    for s in sorted(safe_senders):
        f.write(s + '\n')

print("Safe senders extracted from training data:")
for s in sorted(safe_senders):
    print(f"  {s}")

# Copy vocab.txt
import shutil
shutil.copy(os.path.join(MODEL_DIR, 'vocab.txt'), './vocab.txt')
print(f"\n✓ vocab.txt copied ({os.path.getsize('./vocab.txt')/1024:.0f} KB)")
print(f"✓ safe_senders.txt ({len(safe_senders)} senders)")

In [ ]:
# Verify TFLite model with test examples
print("Verifying TFLite model...\n")

interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()
inp_details = interpreter.get_input_details()
out_details = interpreter.get_output_details()

test_msgs = [
    ("MobileMoney: Payment made for GHS 30.00 to xxx. Transaction ID: 78064510524.", "Safe"),
    ("MTN: Y'ello! You have successfully purchased GHS10 Data Bundle.", "Safe"),
    ("Unknown: Your package delivery has been suspended. Verify: https://turviza.com/ns", "Threat"),
    ("Unknown: URGENT: Your bank account will be suspended! Click http://bit.ly/verify", "Threat"),
    ("Telecel: Enjoy 20% off at Papaye! Dial *533# to claim your discount now.", "Safe"),
    ("NLA: TODAY NLA LOTTO NUMBERS WIN some CASH on Afriluck!", "Threat"),
    ("Unknown: Our company has noticed your information. Contact: wa.me/18396665230", "Threat"),
]

print(f"{'Message':<65} {'Expected':<8} {'Pred':<8} {'Conf'}")
print('-' * 95)

for msg, expected in test_msgs:
    enc = tokenizer(msg, padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors='np')
    interpreter.set_tensor(inp_details[0]['index'], enc['input_ids'].astype(np.int32))
    interpreter.set_tensor(inp_details[1]['index'], enc['attention_mask'].astype(np.int32))
    interpreter.invoke()
    probs = interpreter.get_tensor(out_details[0]['index'])[0]
    pred = 'Threat' if np.argmax(probs) == 1 else 'Safe'
    ok = '✓' if pred == expected else '✗'
    print(f"  {msg[:62]:<62} {expected:<8} {pred:<8} {np.max(probs)*100:.1f}% {ok}")

In [ ]:
# Download all files for Android integration
from google.colab import files

print("Files for android_app/app/src/main/assets/:\n")
files.download(TFLITE_PATH)
files.download('./vocab.txt')
files.download('./safe_senders.txt')
print(f"  1. smishguard_model.tflite ({size_mb:.1f} MB)")
print(f"  2. vocab.txt")
print(f"  3. safe_senders.txt")

print("\nFiles for capstone presentation:\n")
files.download('confusion_matrix.png')
files.download('training_history.png')
print(f"  4. confusion_matrix.png")
print(f"  5. training_history.png")

---
## Summary & Android Integration

### Files to place in `android_app/app/src/main/assets/`:
| File | Purpose |
|------|---------|
| `smishguard_model.tflite` | Fine-tuned DistilBERT binary classifier |
| `vocab.txt` | WordPiece tokenizer vocabulary (30,522 tokens) |
| `safe_senders.txt` | Known safe sender names from training data |

### On-device classification pipeline:
```
SMS arrives → Extract sender name
    ↓
[1] Is sender in user's phone contacts? → SAFE (skip model)
    ↓
[2] Is sender in safe_senders.txt / safe_chats whitelist? → SAFE (skip model)
    ↓
[3] Run DistilBERT: Safe (0) vs Threat (1)
    ↓
[4] If Threat → regex rules classify as SPAM or FRAUD
    • Spam: lottery, betting, gambling keywords
    • Fraud: urgency, fake URLs, delivery scams, work scams
```

### Visual indicators in SmishGuard app:
- **No dot** — Safe message
- **Blue dot** — Spam message
- **Orange dot** — Fraud message